In [1]:
import numpy as np
import random
import pandas as pd
from tensorflow.keras.datasets import cifar10
from tensorflow.keras.utils import to_categorical
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Conv2D, MaxPooling2D, Flatten, Dense
from tensorflow.keras.optimizers import SGD, Adam

# Load data
(X_train, y_train), (X_test, y_test) = cifar10.load_data()

X_train = X_train / 255.0
X_test = X_test / 255.0

y_train = to_categorical(y_train, 10)
y_test = to_categorical(y_test, 10)

# Model builder
def create_model(filters, kernel_size):
    model = Sequential([
        Conv2D(filters, kernel_size, activation='relu', input_shape=(32,32,3)),
        MaxPooling2D(),
        Conv2D(filters*2, kernel_size, activation='relu'),
        MaxPooling2D(),
        Flatten(),
        Dense(128, activation='relu'),
        Dense(10, activation='softmax')
    ])
    return model

results = []

# Random Search
for i in range(5):   # try 5 configs
    lr = random.choice([0.01, 0.001])
    batch_size = random.choice([32, 64])
    filters = random.choice([32, 64])
    kernel = random.choice([(3,3), (5,5)])
    opt_name = random.choice(["sgd", "adam"])
    
    if opt_name == "sgd":
        optimizer = SGD(learning_rate=lr)
    else:
        optimizer = Adam(learning_rate=lr)
    
    print(f"\nRun {i+1}: LR={lr}, Batch={batch_size}, Filters={filters}, Kernel={kernel}, Opt={opt_name}")
    
    model = create_model(filters, kernel)
    model.compile(optimizer=optimizer, loss='categorical_crossentropy', metrics=['accuracy'])
    
    history = model.fit(
        X_train, y_train,
        epochs=5,
        batch_size=batch_size,
        validation_data=(X_test, y_test),
        verbose=0
    )
    
    acc = history.history['val_accuracy'][-1]
    
    results.append({
        "LR": lr,
        "Batch": batch_size,
        "Filters": filters,
        "Kernel": kernel,
        "Optimizer": opt_name,
        "Val Acc": round(acc, 4)
    })

df = pd.DataFrame(results)
print("\nBest Config:")
print(df.sort_values(by="Val Acc", ascending=False))

2026-04-14 13:15:57.692328: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1776172558.050670      22 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1776172558.160433      22 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1776172559.012639      22 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1776172559.012680      22 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1776172559.012683      22 computation_placer.cc:177] computation placer alr

170498071/170498071 ━━━━━━━━━━━━━━━━━━━━ 7s 0us/step


I0000 00:00:1776172601.726850      22 gpu_device.cc:2019] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 15511 MB memory:  -> device: 0, name: Tesla P100-PCIE-16GB, pci bus id: 0000:00:04.0, compute capability: 6.0
/usr/local/lib/python3.12/dist-packages/keras/src/layers/convolutional/base_conv.py:113: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)



Run 1: LR=0.001, Batch=64, Filters=64, Kernel=(3, 3), Opt=adam


I0000 00:00:1776172606.404751      64 service.cc:152] XLA service 0x7bd1c400bf60 initialized for platform CUDA (this does not guarantee that XLA will be used). Devices:
I0000 00:00:1776172606.404785      64 service.cc:160]   StreamExecutor device (0): Tesla P100-PCIE-16GB, Compute Capability 6.0
I0000 00:00:1776172606.769541      64 cuda_dnn.cc:529] Loaded cuDNN version 91002
I0000 00:00:1776172609.336347      64 device_compiler.h:188] Compiled cluster using XLA!  This line is logged at most once for the lifetime of the process.



Run 2: LR=0.01, Batch=32, Filters=32, Kernel=(5, 5), Opt=sgd

Run 3: LR=0.01, Batch=64, Filters=64, Kernel=(5, 5), Opt=adam

Run 4: LR=0.01, Batch=64, Filters=64, Kernel=(3, 3), Opt=sgd

Run 5: LR=0.001, Batch=64, Filters=32, Kernel=(5, 5), Opt=adam

Best Config:
      LR  Batch  Filters  Kernel Optimizer  Val Acc
0  0.001     64       64  (3, 3)      adam   0.6927
4  0.001     64       32  (5, 5)      adam   0.6776
1  0.010     32       32  (5, 5)       sgd   0.5387
3  0.010     64       64  (3, 3)       sgd   0.4533
2  0.010     64       64  (5, 5)      adam   0.1000
